# Build Player vs Team Stats - Silver Layer

Aggregates player performance metrics against specific opponent teams from StatsBomb event data.

## Target Schema
* **player_id** (INT, PK)
* **player_name** (STRING)
* **opponent_team_id** (INT, PK)
* **opponent_team_name** (STRING)
* **total_appearances** (INT) - Matches played against this opponent
* **goals** (INT) - Goals scored against this opponent
* **shots** (INT) - Shots taken against this opponent
* **xg** (DOUBLE) - Expected goals against this opponent
* **last_updated** (TIMESTAMP) - Timestamp of last update

## Use Cases
* Analyze player performance trends against specific teams
* Identify players who excel/struggle against certain opponents
* Support head-to-head matchup predictions

In [0]:
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from config.paths import EVENTS_BRONZE, LINEUPS_BRONZE

# Initialize Spark session
spark = SparkSession.builder.appName("BuildPlayerVsTeam").getOrCreate()

print("=" * 80)
print("Building Player vs Team Stats - Silver Layer")
print("=" * 80)

In [0]:
# Read events and lineups from bronze layer
print("\n[1/6] Reading bronze layer data...")
df_events = spark.read.parquet(EVENTS_BRONZE)
df_lineups = spark.read.parquet(LINEUPS_BRONZE)

print(f"   Total events: {df_events.count():,}")
print(f"   Total lineup records: {df_lineups.count():,}")

# Filter for player events only
df_player_events = df_events.filter(F.col("player_id").isNotNull())
print(f"   Player events: {df_player_events.count():,}")

In [0]:
# Get unique matches with both teams per match
print("\n[2/6] Identifying opponent teams...")

# Get player's team per match from lineups
df_player_teams = df_lineups.select(
    F.col("match_id"),
    F.col("player_id"),
    F.col("player_name"),
    F.col("team_id").alias("player_team_id")
).distinct()

# Get all teams per match for opponent lookup
df_match_teams = df_lineups.select(
    F.col("match_id"),
    F.col("team_id"),
    F.col("team_name")
).distinct()

# Join to get opponent teams (team_id != player_team_id in same match)
df_opponents = df_player_teams.alias("pt").join(
    df_match_teams.alias("mt"),
    (F.col("pt.match_id") == F.col("mt.match_id")) & 
    (F.col("pt.player_team_id") != F.col("mt.team_id")),
    "inner"
).select(
    F.col("pt.match_id"),
    F.col("pt.player_id"),
    F.col("pt.player_name"),
    F.col("pt.player_team_id"),
    F.col("mt.team_id").alias("opponent_team_id"),
    F.col("mt.team_name").alias("opponent_team_name")
)

print(f"   Player-opponent pairs: {df_opponents.count():,}")

In [0]:
# Extract shot-specific fields from raw_json
print("\n[3/6] Extracting shot metrics...")

df_with_shots = df_player_events.withColumn(
    "shot_xg",
    F.when(
        F.col("event_type_name") == "Shot",
        F.get_json_object(F.col("raw_json"), "$.shot.statsbomb_xg").cast(DoubleType())
    ).otherwise(F.lit(None))
).withColumn(
    "shot_outcome",
    F.when(
        F.col("event_type_name") == "Shot",
        F.get_json_object(F.col("raw_json"), "$.shot.outcome.name")
    ).otherwise(F.lit(None))
)

print("   Shot metrics extracted ✓")

In [0]:
# Join events with opponent team data
print("\n[4/6] Joining events with opponent data...")

# Drop player_name from df_opponents to avoid duplicate column after join
df_opponents_clean = df_opponents.drop("player_name")

df_events_with_opponents = df_with_shots.join(
    df_opponents_clean,
    ["match_id", "player_id"],
    "inner"
)

print(f"   Events with opponent context: {df_events_with_opponents.count():,}")

In [0]:
# Aggregate by player and opponent team
print("\n[5/6] Aggregating player vs team statistics...")

df_player_vs_team = df_events_with_opponents.groupBy(
    "player_id",
    "player_name",
    "opponent_team_id",
    "opponent_team_name"
).agg(
    # Match appearances against this opponent
    F.countDistinct("match_id").alias("total_appearances"),
    
    # Shot metrics against this opponent
    F.sum(
        F.when(F.col("shot_outcome") == "Goal", 1).otherwise(0)
    ).alias("goals"),
    
    F.sum(
        F.when(F.col("event_type_name") == "Shot", 1).otherwise(0)
    ).alias("shots"),
    
    F.sum(
        F.coalesce(F.col("shot_xg"), F.lit(0.0))
    ).alias("xg")
).withColumn(
    "last_updated",
    F.current_timestamp()
)

# Cast to match target schema
df_final = df_player_vs_team.select(
    F.col("player_id").cast("int"),
    F.col("player_name"),
    F.col("opponent_team_id").cast("int"),
    F.col("opponent_team_name"),
    F.col("total_appearances").cast("int"),
    F.col("goals").cast("int"),
    F.col("shots").cast("int"),
    F.col("xg").cast("double"),
    F.col("last_updated")
)

print(f"   Player-opponent combinations: {df_final.count():,}")
print("   Aggregation complete ✓")

In [0]:
# Show sample data
print("\n[6/6] Sample Data:")
display(df_final.orderBy(F.desc("goals")).limit(20))

In [0]:
# Show summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\nTotal player-opponent combinations: {df_final.count():,}")
print(f"Unique players: {df_final.select('player_id').distinct().count():,}")
print(f"Unique opponent teams: {df_final.select('opponent_team_id').distinct().count():,}")

# Top performers against specific opponents
print("\n" + "=" * 80)
print("TOP 10 PLAYER PERFORMANCES VS SPECIFIC OPPONENTS")
print("=" * 80)

df_top_performances = df_final.filter(F.col("total_appearances") >= 2) \
    .orderBy(F.desc("goals")) \
    .limit(10)

display(df_top_performances)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

In [0]:
# Write to silver layer (Delta table)
print("\nWriting to silver layer...")

target_table = "matchpulse.silver.player_vs_team"

df_final.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✓ Successfully wrote to {target_table}")
print(f"✓ Total records: {df_final.count():,}")

In [0]:
%sql
-- Verify the table was created successfully
DESCRIBE TABLE EXTENDED matchpulse.silver.player_vs_team;

In [0]:
%sql
-- Query top performers against specific opponents
SELECT 
    player_id,
    player_name,
    opponent_team_id,
    opponent_team_name,
    total_appearances,
    goals,
    shots,
    ROUND(xg, 2) as xg,
    ROUND(goals / total_appearances, 2) as goals_per_match,
    ROUND(xg / NULLIF(shots, 0), 3) as xg_per_shot
FROM matchpulse.silver.player_vs_team
WHERE total_appearances >= 3
ORDER BY goals DESC
LIMIT 20;

In [0]:
df = spark.table("matchpulse.silver.player_vs_team")

In [0]:
from pyspark.sql import functions as F

df_metrics = (
    df
    .withColumn(
        "shot_conversion",
        F.round(
            F.try_divide(F.col("goals"), F.col("shots")) * 100,
            2
        )
    )
    .withColumn(
        "xg_per_shot",
        F.round(
            F.try_divide(F.col("xg"), F.col("shots")),
            3
        )
    )
    .withColumn(
        "goals_minus_xg",
        F.round(
            F.col("goals") - F.col("xg"),
            2
        )
    )
)

In [0]:
df_filtered = (
    df_metrics
    .filter(F.col("shots") >= 20)
    .filter(F.col("total_appearances") >= 3)
)

In [0]:
pdf = df_filtered.toPandas()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Messi only
messi = pdf[pdf["player_name"].str.contains("Messi")]

# Sort
messi = messi.sort_values("goals", ascending=True)

plt.figure(figsize=(12, 10))

# Lines
plt.hlines(
    y=messi["opponent_team_name"],
    xmin=0,
    xmax=messi["goals"],
    color='gray',
    alpha=0.6
)

# Dots
plt.scatter(
    messi["goals"],
    messi["opponent_team_name"],
    s=messi["shots"] * 2,
    c=messi["xg_per_shot"],
    cmap="viridis"
)

plt.title(
    "Lionel Messi's Favorite Opponents",
    fontsize=20
)

plt.xlabel("Goals Scored")
plt.ylabel("Opponent")

plt.colorbar(label="xG per Shot")

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.facecolor'] = 'white'

# Create figure with multiple subplots
fig = plt.figure(figsize=(20, 24))

# ============================================================================
# 1. HEATMAP: Top 15 Players vs Top 10 Opponents (Goals)
# ============================================================================
ax1 = plt.subplot(4, 2, 1)

# Get top 15 players by total goals (limit Messi to top 3 opponents)
messi_top3 = pdf[pdf["player_name"].str.contains("Messi")].nlargest(3, "goals")
others = pdf[~pdf["player_name"].str.contains("Messi")]
top_players_data = pd.concat([messi_top3, others]).groupby("player_name")["goals"].sum().nlargest(15).index

# Get top 10 opponents
top_opponents = pdf.groupby("opponent_team_name")["goals"].sum().nlargest(10).index

# Create pivot table
heatmap_data = pdf[pdf["player_name"].isin(top_players_data) & 
                    pdf["opponent_team_name"].isin(top_opponents)].pivot_table(
    values="goals",
    index="player_name",
    columns="opponent_team_name",
    aggfunc="sum",
    fill_value=0
)

sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd", 
            cbar_kws={'label': 'Goals'}, ax=ax1)
ax1.set_title("Goals Heatmap: Top Players vs Top Opponents", fontsize=14, fontweight='bold')
ax1.set_xlabel("Opponent Team")
ax1.set_ylabel("Player")
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ============================================================================
# 2. SCATTER: xG vs Actual Goals (Overperformers/Underperformers)
# ============================================================================
ax2 = plt.subplot(4, 2, 2)

# Highlight Messi with different color
is_messi = pdf["player_name"].str.contains("Messi")
colors = np.where(is_messi, 'red', 'steelblue')
alphas = np.where(is_messi, 0.8, 0.6)

for i, row in pdf.iterrows():
    ax2.scatter(row["xg"], row["goals"], 
               s=row["shots"]*3, 
               c=colors[i], 
               alpha=alphas[i],
               edgecolors='black',
               linewidth=0.5)

# Add diagonal line (perfect xG match)
max_val = max(pdf["xg"].max(), pdf["goals"].max())
ax2.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Perfect xG match')

# Label top overperformers (goals > xG + 5)
overperformers = pdf[pdf["goals_minus_xg"] > 5].nlargest(5, "goals_minus_xg")
for _, row in overperformers.iterrows():
    label = row["player_name"].split()[-1][:8] + " vs " + row["opponent_team_name"][:6]
    ax2.annotate(label, (row["xg"], row["goals"]), 
                fontsize=7, alpha=0.7, 
                xytext=(5, 5), textcoords='offset points')

ax2.set_xlabel("Expected Goals (xG)", fontsize=11)
ax2.set_ylabel("Actual Goals", fontsize=11)
ax2.set_title("Goal Performance vs Expected (xG)\nBubble size = Shots taken", 
             fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# ============================================================================
# 3. BAR CHART: Top 20 Player-Opponent Combinations by Goals
# ============================================================================
ax3 = plt.subplot(4, 2, 3)

# Limit Messi to top 4 opponents
messi_top4 = pdf[pdf["player_name"].str.contains("Messi")].nlargest(4, "goals")
others_top = pdf[~pdf["player_name"].str.contains("Messi")].nlargest(16, "goals")
top_combinations = pd.concat([messi_top4, others_top]).nlargest(20, "goals")

top_combinations["label"] = (top_combinations["player_name"].str.split().str[-1] + 
                             " vs " + top_combinations["opponent_team_name"])

bars = ax3.barh(range(len(top_combinations)), top_combinations["goals"], 
                color=['crimson' if 'Messi' in name else 'steelblue' 
                       for name in top_combinations["player_name"]])
ax3.set_yticks(range(len(top_combinations)))
ax3.set_yticklabels(top_combinations["label"], fontsize=9)
ax3.set_xlabel("Goals", fontsize=11)
ax3.set_title("Top 20 Player-Opponent Rivalries by Goals", fontsize=14, fontweight='bold')
ax3.invert_yaxis()

# Add value labels
for i, (idx, row) in enumerate(top_combinations.iterrows()):
    ax3.text(row["goals"] + 0.3, i, f"{int(row['goals'])}", 
            va='center', fontsize=8)

# ============================================================================
# 4. BUBBLE CHART: Shot Efficiency (Conversion Rate vs Volume)
# ============================================================================
ax4 = plt.subplot(4, 2, 4)

# Filter for players with meaningful data
efficiency_data = pdf[pdf["shots"] >= 30].copy()

scatter = ax4.scatter(efficiency_data["shots"], 
                     efficiency_data["shot_conversion"],
                     s=efficiency_data["goals"]*10,
                     c=efficiency_data["xg_per_shot"],
                     cmap='plasma',
                     alpha=0.6,
                     edgecolors='black',
                     linewidth=0.5)

plt.colorbar(scatter, ax=ax4, label='xG per Shot')

# Label top converters
top_converters = efficiency_data.nlargest(5, "shot_conversion")
for _, row in top_converters.iterrows():
    label = row["player_name"].split()[-1][:8]
    ax4.annotate(label, (row["shots"], row["shot_conversion"]),
                fontsize=8, alpha=0.7,
                xytext=(3, 3), textcoords='offset points')

ax4.set_xlabel("Total Shots", fontsize=11)
ax4.set_ylabel("Shot Conversion Rate (%)", fontsize=11)
ax4.set_title("Shot Efficiency Analysis\nBubble size = Goals scored", 
             fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# ============================================================================
# 5. DISTRIBUTION: Goals per Match Distribution
# ============================================================================
ax5 = plt.subplot(4, 2, 5)

pdf["goals_per_match"] = pdf["goals"] / pdf["total_appearances"]

# Split data
messi_data = pdf[pdf["player_name"].str.contains("Messi")]["goals_per_match"]
other_data = pdf[~pdf["player_name"].str.contains("Messi")]["goals_per_match"]

ax5.hist(other_data, bins=30, alpha=0.7, color='steelblue', label='Other Players', edgecolor='black')
ax5.hist(messi_data, bins=15, alpha=0.7, color='crimson', label='Messi', edgecolor='black')

ax5.axvline(other_data.mean(), color='steelblue', linestyle='--', linewidth=2, 
           label=f'Others Mean: {other_data.mean():.2f}')
ax5.axvline(messi_data.mean(), color='crimson', linestyle='--', linewidth=2,
           label=f'Messi Mean: {messi_data.mean():.2f}')

ax5.set_xlabel("Goals per Match", fontsize=11)
ax5.set_ylabel("Frequency", fontsize=11)
ax5.set_title("Distribution of Goals per Match", fontsize=14, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# ============================================================================
# 6. VIOLIN PLOT: xG Performance by Top Players
# ============================================================================
ax6 = plt.subplot(4, 2, 6)

# Get top 10 players by total goals
top_10_players = pdf.groupby("player_name")["goals"].sum().nlargest(10).index
violin_data = pdf[pdf["player_name"].isin(top_10_players)].copy()

# Sort by median goals_minus_xg
player_order = violin_data.groupby("player_name")["goals_minus_xg"].median().sort_values(ascending=False).index

sns.violinplot(data=violin_data, y="player_name", x="goals_minus_xg",
              order=player_order, palette="Set2", ax=ax6)

ax6.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1)
ax6.set_xlabel("Goals vs xG Difference", fontsize=11)
ax6.set_ylabel("Player", fontsize=11)
ax6.set_title("Performance vs Expected: Top 10 Goal Scorers", fontsize=14, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='x')

# ============================================================================
# 7. STACKED BAR: Shot Volume Breakdown for Top Opponents
# ============================================================================
ax7 = plt.subplot(4, 2, 7)

# Get top 10 opponents by total shots faced
top_opponents_shots = pdf.groupby("opponent_team_name")["shots"].sum().nlargest(10)
shot_breakdown = pdf[pdf["opponent_team_name"].isin(top_opponents_shots.index)].groupby("opponent_team_name").agg({
    "shots": "sum",
    "goals": "sum"
}).reset_index()

shot_breakdown["missed"] = shot_breakdown["shots"] - shot_breakdown["goals"]
shot_breakdown = shot_breakdown.sort_values("shots", ascending=True)

ax7.barh(shot_breakdown["opponent_team_name"], shot_breakdown["goals"], 
        label='Goals', color='#2ecc71')
ax7.barh(shot_breakdown["opponent_team_name"], shot_breakdown["missed"],
        left=shot_breakdown["goals"], label='Missed', color='#95a5a6')

ax7.set_xlabel("Total Shots", fontsize=11)
ax7.set_ylabel("Opponent Team", fontsize=11)
ax7.set_title("Shot Volume & Conversion: Top 10 Opponents Faced", fontsize=14, fontweight='bold')
ax7.legend()
ax7.grid(True, alpha=0.3, axis='x')

# ============================================================================
# 8. BOX PLOT: xG per Shot Distribution by Shot Volume Categories
# ============================================================================
ax8 = plt.subplot(4, 2, 8)

# Categorize by shot volume
pdf["shot_category"] = pd.cut(pdf["shots"], 
                              bins=[0, 30, 50, 100, 500],
                              labels=['20-30', '31-50', '51-100', '100+'])

sns.boxplot(data=pdf, x="shot_category", y="xg_per_shot", 
           palette="viridis", ax=ax8)

# Add swarm plot for individual points
sns.swarmplot(data=pdf, x="shot_category", y="xg_per_shot",
             color='red', alpha=0.3, size=3, ax=ax8)

ax8.set_xlabel("Shot Volume Category", fontsize=11)
ax8.set_ylabel("xG per Shot", fontsize=11)
ax8.set_title("Shot Quality Distribution by Volume", fontsize=14, fontweight='bold')
ax8.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("VISUALIZATION SUMMARY")
print("="*80)
print(f"Total records visualized: {len(pdf)}")
print(f"Unique players: {pdf['player_name'].nunique()}")
print(f"Unique opponents: {pdf['opponent_team_name'].nunique()}")
print(f"\nTop 5 Players by Goals:")
print(pdf.groupby('player_name')['goals'].sum().nlargest(5))
print(f"\nTop 5 Opponents Faced (by total shots):")
print(pdf.groupby('opponent_team_name')['shots'].sum().nlargest(5))

In [0]:
top_players = (
    pdf.groupby("player_name")["goals"]
    .sum()
    .nlargest(10)
    .index
)

top_teams = (
    pdf.groupby("opponent_team_name")["goals"]
    .sum()
    .nlargest(10)
    .index
)

heatmap_df = pdf[
    (pdf["player_name"].isin(top_players)) &
    (pdf["opponent_team_name"].isin(top_teams))
]

pivot = heatmap_df.pivot_table(
    values="goals",
    index="player_name",
    columns="opponent_team_name",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(14, 8))

sns.heatmap(
    pivot,
    annot=True,
    cmap="YlOrRd",
    fmt=".0f"
)

plt.title(
    "Goals Heatmap: Players vs Opponents",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(14, 10))

sns.scatterplot(
    data=pdf,
    x="xg",
    y="goals",
    size="shots",
    hue="goals_minus_xg",
    palette="coolwarm",
    sizes=(50, 500),
    alpha=0.8
)

max_val = max(pdf["xg"].max(), pdf["goals"].max())

plt.plot(
    [0, max_val],
    [0, max_val],
    linestyle='--',
    color='black'
)

# Label top overperformers
top = pdf.nlargest(10, "goals_minus_xg")

for _, row in top.iterrows():

    plt.text(
        row["xg"] + 0.2,
        row["goals"] + 0.2,
        row["player_name"].split()[-1],
        fontsize=8
    )

plt.title("Goals vs xG", fontsize=20)

plt.xlabel("Expected Goals")
plt.ylabel("Actual Goals")

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create goals per match metric
pdf["goals_per_match"] = (
    pdf["goals"] / pdf["total_appearances"]
)

# Split Messi vs Others
messi_data = pdf[
    pdf["player_name"].str.contains("Messi")
]["goals_per_match"]

other_data = pdf[
    ~pdf["player_name"].str.contains("Messi")
]["goals_per_match"]

# Plot
plt.figure(figsize=(12, 8))

plt.hist(
    other_data,
    bins=30,
    alpha=0.7,
    color='steelblue',
    label='Other Players',
    edgecolor='black'
)

plt.hist(
    messi_data,
    bins=15,
    alpha=0.7,
    color='crimson',
    label='Messi',
    edgecolor='black'
)

# Mean lines
plt.axvline(
    other_data.mean(),
    color='steelblue',
    linestyle='--',
    linewidth=2,
    label=f'Others Mean: {other_data.mean():.2f}'
)

plt.axvline(
    messi_data.mean(),
    color='crimson',
    linestyle='--',
    linewidth=2,
    label=f'Messi Mean: {messi_data.mean():.2f}'
)

# Labels
plt.xlabel("Goals per Match", fontsize=12)
plt.ylabel("Frequency", fontsize=12)

plt.title(
    "Distribution of Goals per Match",
    fontsize=18,
    fontweight='bold'
)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()